# <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;  " > 练习实验：基于内容过滤的深度学习

在本练习中，你将用神经网络实现基于内容的过滤，构建一个电影推荐系统。 

# 大纲 <img align="left" src="./images/film_reel.png"     style=" width:40px;  " >
- [ 1 - 包](#1)
- [ 2 - 电影评分数据集](#2)
  - [ 2.1 用神经网络进行基于内容的过滤](#2.1)
  - [ 2.2 准备训练数据](#2.2)
- [ 3 - 用于基于内容过滤的神经网络](#3)
  - [ 3.1 预测](#3.1)
    - [ 练习 1](#ex01)
- [ 4 - 恭喜！](#4)


<a name="1"></a>
## 1 - 包 <img align="left" src="./images/movie_camera.png"     style=" width:40px;  ">
我们将使用熟悉的包：NumPy、TensorFlow，以及 [scikit-learn](https://scikit-learn.org/stable/) 的实用例程。还会用 [tabulate](https://pypi.org/project/tabulate/) 来整齐地打印表格，用 [Pandas](https://pandas.pydata.org/) 来组织表格数据。

In [1]:
import numpy as np
import numpy.ma as ma
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - 电影评分数据集 <img align="left" src="./images/film_rating.png" style=" width:40px;" >
该数据集源自 [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) 数据集。 

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集包含 600 个用户对 9000 部电影的评分，评分范围是 0.5 到 5，步长为 0.5。数据集经过缩减，聚焦于 2000 年以来的电影和热门类型。缩减后的数据集有 $n_u = 395$ 个用户和 $n_m= 694$ 部电影。对每部电影，数据集提供片名、上映年份以及一个或多个类型。例如“Toy Story 3”于 2010 年上映，有多个类型：“Adventure|Animation|Children|Comedy|Fantasy|IMAX”。该数据集除了评分之外几乎不包含用户信息。此数据集用于为下面描述的神经网络创建训练向量。 

<a name="2.1"></a>
### 2.1 用神经网络进行基于内容的过滤

在协同过滤实验中，你生成了两个向量——用户向量和物品/电影向量，二者的点积用于预测评分。这些向量完全由评分推导而来。   

基于内容的过滤同样生成用户向量和电影特征向量，但它认识到可能还有关于用户和/或电影的其他信息可用于改进预测。这些附加信息被提供给神经网络，由网络生成用户向量和电影向量，如下图所示。
<figure>
    <center> <img src="./images/RecSysNN.png"   style="width:500px;height:280px;" ></center>
</figure>
提供给网络的电影内容是原始数据和一些“工程特征”的组合。回忆一下课程 1 第 2 周实验 4 中关于特征工程的讨论。原始特征是电影上映年份和以独热向量表示的电影类型，共有 14 个类型。工程特征是由用户评分推导出的平均评分。具有多个类型的电影，每个类型对应一个训练向量。 

用户内容仅由工程特征组成。为每个用户计算其按类型划分的平均评分。此外，用户 id、评分数和评分均值也可用，但不包含在训练或预测内容中。它们有助于解读数据。

训练集由数据集中用户做出的所有评分组成。用户向量和电影/物品向量一起作为训练集送入上述网络。对同一用户评过的所有电影，用户向量都是相同的。 

下面，让我们加载并展示一些数据。

In [2]:
# 加载数据，设置配置变量
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items
scaledata = True  # applies the standard scalar to data if true
print(f"Number of training vectors: {len(item_train)}")

Number of training vectors: 58187


部分用户和物品/电影特征不用于训练。下面，方括号“[]”中的特征（如“user id”“rating count”“rating ave”）在模型训练和使用时都不包含在内。注意，对同一用户评过的所有电影，用户向量都是相同的。

In [3]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9


In [4]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
6874,2003,4.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8798,2004,3.8,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [5]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [4.  4.  4.  3.5 3.5]


上面可以看到，电影 6874 是一部 2003 年上映的动作片。用户 2 对动作片的平均评分为 3.9。此外，电影 6874 同时被列在犯罪（Crime）和惊悚（Thriller）类型下。MovieLens 用户给这部电影的平均评分是 4。一个训练样本由两张表中各取一行以及 y_train 中的一个评分组成。

<a name="2.2"></a>
### 2.2 准备训练数据
回忆一下，在课程 1 第 2 周，你探索过用特征缩放来改善收敛。我们将使用 [scikit learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 缩放输入特征。这在课程 1 第 2 周实验 5 中使用过。下面还展示了 inverse_transform，用于还原出原始输入。

In [6]:
# 缩放训练数据
if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    scalerItem = StandardScaler()
    scalerItem.fit(item_train)
    item_train = scalerItem.transform(item_train)

    scalerUser = StandardScaler()
    scalerUser.fit(user_train)
    user_train = scalerUser.transform(user_train)

    print(np.allclose(item_train_save, scalerItem.inverse_transform(item_train)))
    print(np.allclose(user_train_save, scalerUser.inverse_transform(user_train)))

True
True


为了能够评估结果，我们将按课程 2 第 3 周讨论的方法把数据划分为训练集和测试集。这里使用 [sklearn train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来划分并打乱数据。注意，把初始随机状态设为相同值可以确保 item、user 和 y 被以相同方式打乱。

In [7]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

movie/item training data shape: (46549, 17)


movie/item test  data shape: (11638, 17)


缩放并打乱后的数据现在均值为零。

In [8]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
1,0,0.6,0.7,0.6,0.6,0.7,0.7,0.5,0.7,0.2,0.3,0.3,0.5,0.5,0.8,0.5
0,0,1.6,1.5,1.7,0.9,1.0,1.4,0.8,-1.2,1.2,1.2,1.6,0.9,1.4,1.2,1.0
0,0,0.8,0.6,0.7,0.5,0.6,0.6,0.3,-1.2,0.7,0.8,0.9,0.6,0.2,0.6,0.6
1,0,-0.1,0.2,-0.1,0.3,0.7,0.3,0.2,1.0,-0.5,-0.7,-2.1,0.5,0.7,0.3,0.0
-1,0,-1.3,-0.8,-0.8,0.1,-0.1,-1.1,-0.9,-1.2,-1.5,-0.6,-0.5,-0.6,-0.9,-0.4,-0.9


使用 Min Max Scaler 缩放目标评分，把目标缩放到 -1 和 1 之间。我们使用 scikit-learn，因为它带有 inverse_transform。[scikit learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [9]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train = scaler.transform(y_train.reshape(-1, 1))
ynorm_test = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train.shape, ynorm_test.shape)

(46549, 1) (11638, 1)


<a name="3"></a>
## 3 - 用于基于内容过滤的神经网络
现在，让我们按上图构建一个神经网络。它由两个网络组成，通过点积结合在一起。你将构建这两个网络。本例中它们是完全相同的。注意，这两个网络并不需要相同。如果用户内容比电影内容大得多，你可以选择让用户网络比电影网络更复杂。本例中两者内容相似，所以网络相同。

- 使用 Keras sequential 模型
    - 第一层是 256 个单元、relu 激活的 dense 层。
    - 第二层是 128 个单元、relu 激活的 dense 层。
    - 第三层是 `num_outputs` 个单元、线性（或无）激活的 dense 层。   
    
网络的其余部分已经提供。提供的代码没有使用 Keras sequential 模型，而是使用 Keras [函数式 api](https://keras.io/guides/functional_api/)。这种形式在组件互联方式上更灵活。


In [10]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    # ## START CODE HERE ###   
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs, activation='linear'),
    # ## END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    # ## START CODE HERE ###     
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs, activation='linear'),
    # ## END CODE HERE ###  
])

# 创建用户输入并指向基础网络
input_user = tf.keras.layers.Input(shape=(num_user_features,))
vu = user_NN(input_user)
vu = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vu)

# 创建物品输入并指向基础网络
input_item = tf.keras.layers.Input(shape=(num_item_features,))
vm = item_NN(input_item)
vm = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vm)

# 计算两个向量 vu 和 vm 的点积
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# 指定模型的输入和输出
model = Model([input_user, input_item], output)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32)        │     40,864 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 32)        │     41,376 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 32)        │          0 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 32)        │          0 │ sequential_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ lambda[0][0],     │
│                     │                   │            │ lambda_1[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 82,240 (321.25 KB)

 Trainable params: 82,240 (321.25 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# 公开测试
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

All tests passed!
All tests passed!


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  可以按如下方式创建 relu 激活的 dense 层。
    
```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看答案</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```
</details>
</details>

    


我们将使用均方误差损失和 Adam 优化器。

In [12]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [13]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], ynorm_train, epochs=30)

Epoch 1/30


   1/1455 ━━━━━━━━━━━━━━━━━━━━ 44:30 2s/step - loss: 0.4612

  22/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1808  

  42/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1608

  63/1455 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1484

...

1365/1455 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0984

1388/1455 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0985

1411/1455 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0986

1438/1455 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0985

1455/1455 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0984


评估模型，确定其在测试数据上的损失。它与训练损失相当，表明模型没有明显过拟合训练数据。

In [14]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], ynorm_test)

  1/364 ━━━━━━━━━━━━━━━━━━━━ 1:07 186ms/step - loss: 0.1296

 27/364 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1025    

 49/364 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1027

 71/364 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1034

 97/364 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1047

...

249/364 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1063

290/364 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1065

328/364 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1063

364/364 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1062


0.10618384182453156

<a name="3.1"></a>
### 3.1 预测
下面，你将在多种场景下使用模型进行预测。 
#### 为新用户预测
首先，我们创建一个新用户，让模型为该用户推荐电影。在示例用户内容上试过之后，可以随意修改用户内容以匹配你自己的偏好，看看模型会推荐什么。注意，评分在 0.5 到 5.0 之间（含端点），以 0.5 为步长。

In [15]:
new_user_id = 5000
new_rating_ave = 1.0
new_action = 1.0
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 5
new_crime = 1
new_documentary = 1
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 5
new_scifi = 5
new_thriller = 1
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])


让我们看看为新用户推荐的高分电影。回忆一下，用户向量中偏好的类型是喜剧（Comedy）和爱情（Romance）。
下面，我们将使用一组电影/物品向量 `item_vecs`，其中训练/测试集的每部电影都有一个向量。将其与上面的用户向量匹配，用缩放后的向量为我们上面的新用户预测所有电影的评分。

In [16]:
# 生成并复制用户向量，使其数量与数据集中的电影数量一致。
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# 缩放向量并对所有电影进行预测。返回按评分排序的结果。
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, scalerUser, scalerItem, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

 1/59 ━━━━━━━━━━━━━━━━━━━━ 5s 89ms/step

27/59 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


y_p,movie id,rating ave,title,genres
4.84902,5015,3.30556,Monster's Ball (2001),Drama|Romance
4.84757,4299,3.34146,"Knight's Tale, A (2001)",Action|Comedy|Romance
4.84453,6218,3.3,Bend It Like Beckham (2002),Comedy|Drama|Romance
4.8404,5299,3.26667,My Big Fat Greek Wedding (2002),Comedy|Romance
4.83939,5957,3.26316,Two Weeks Notice (2002),Comedy|Romance
4.83639,6953,3.3,21 Grams (2003),Crime|Drama|Mystery|Romance|Thriller
4.83519,6155,3.27586,How to Lose a Guy in 10 Days (2003),Comedy|Romance
4.83362,6373,3.3169,Bruce Almighty (2003),Comedy|Drama|Fantasy|Romance
4.82654,4975,3.42045,Vanilla Sky (2001),Mystery|Romance|Sci-Fi|Thriller
4.82554,6755,3.32143,Bubba Ho-tep (2002),Comedy|Horror


如果你在上面创建了用户，值得注意的是：网络训练时是根据包含**一组**用户类型评分的用户向量来预测评分的。如果真实用户中没有类似的评分组合，仅仅给单个类型打最高分、其余类型打最低分，对网络来说可能没有意义。

#### 为现有用户预测
让我们看看对“用户 36”（数据集中的用户之一）的预测。我们可以把预测评分与该用户的实际评分进行比较。注意，具有多个类型的电影在训练数据中出现多次。例如，《The Time Machine》有三个类型：冒险、动作、科幻。

In [17]:
uid =  36 
# 构建一组用户向量。这是同一个向量经过变换并重复的结果。
user_vecs, y_vecs = get_user_vecs(uid, scalerUser.inverse_transform(user_train), item_vecs, user_to_genre)

# 缩放向量并对所有电影进行预测。返回按评分排序的结果。
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, 
                                                                      scalerUser, scalerItem, scaledata=scaledata)
sorted_y = y_vecs[sorted_index]

# 打印排序后的预测
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, item_features, ivs, uvs, movie_dict, maxcount = 10)

 1/59 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step

46/59 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


y_p,y,user,user genre ave,movie rating ave,title,genres
2.9,3.0,36,3.00,2.86,"Time Machine, The (2002)",Adventure
2.7,3.0,36,3.00,2.86,"Time Machine, The (2002)",Action
2.6,3.0,36,3.00,2.86,"Time Machine, The (2002)",Sci-Fi
2.0,1.0,36,1.50,4.00,"Beautiful Mind, A (2001)",Drama
1.8,1.0,36,1.00,4.00,"Beautiful Mind, A (2001)",Romance
1.8,2.0,36,1.75,3.52,Gangs of New York (2002),Crime
1.8,1.5,36,1.75,3.52,Road to Perdition (2002),Crime
1.5,1.5,36,1.50,3.52,Road to Perdition (2002),Drama
1.4,2.0,36,1.50,3.52,Gangs of New York (2002),Drama


#### 寻找相似物品
上面的神经网络生成两个特征向量：用户特征向量 $v_u$ 和电影特征向量 $v_m$。它们是 32 维向量，取值难以直接解释。然而，相似的物品会有相似的向量。这一信息可用于推荐。例如，如果用户给“Toy Story 3”打了高分，就可以通过挑选电影特征向量相似的电影来进行推荐。

一种相似性度量是两个向量 $ \mathbf{v_m^{(k)}}$ 和 $\mathbf{v_m^{(i)}}$ 之间的平方距离：
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

<a name="ex01"></a>
### 练习 1

编写一个计算平方距离的函数。

In [18]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    # ## START CODE HERE ###     
    d = sum(np.square(a-b))
    # ## END CODE HERE ###     
    return (d)

In [19]:
# 公开测试
test_sq_dist(sq_dist)

All tests passed!


In [20]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

squared distance between a1 and b1: 0.0
squared distance between a2 and b2: 0.030000000000000054
squared distance between a3 and b3: 2


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  虽然求和往往意味着应该用 for 循环，但这里的减法可以在一条语句中逐元素完成。此外，你可以用 np.square 对减法的结果逐元素平方，再用 np.sum 对平方后的元素求和。
    
</details>

    


模型训练完成后，可以一次性计算电影之间的距离矩阵，之后做新推荐时无需重新训练即可重复使用。模型训练好后的第一步，是获取每部电影的电影特征向量 $v_m$。为此，我们将使用训练好的 `item_NN`，并构建一个小模型，让电影向量通过它来生成 $v_m$。

In [21]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features,))    # input layer
vm_m = item_NN(input_item_m)                                       # use the trained item_NN
vm_m = tf.keras.layers.Lambda(lambda x: tf.linalg.l2_normalize(x, axis=1))(vm_m)  # incorporate normalization as was done in the original model
model_m = Model(input_item_m, vm_m)                                
model_m.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 32)             │        41,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,376 (161.62 KB)

 Trainable params: 41,376 (161.62 KB)

 Non-trainable params: 0 (0.00 B)

有了电影模型后，就可以用模型对一组物品/电影向量进行预测，从而创建一组电影特征向量。`item_vecs` 是所有电影向量的集合。回忆一下，同一部电影会按它的每个类型分别作为一个向量出现。这些向量必须经过缩放才能用于训练好的模型。预测的结果是每部电影的 32 维特征向量。

In [22]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

 1/59 ━━━━━━━━━━━━━━━━━━━━ 4s 81ms/step

46/59 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


size of all predicted movie feature vectors: (1883, 32)


现在计算每个电影特征向量与所有其他电影特征向量之间平方距离的矩阵：
<figure>
    <left> <img src="./images/distmatrix.PNG"   style="width:400px;height:225px;" ></center>
</figure>

然后，通过找每行的最小值即可找到最接近的电影。我们将使用 [numpy 掩码数组](https://numpy.org/doc/1.21/user/tutorial-ma.html)来避免选中同一部电影。对角线上的掩码值不会参与计算。

In [23]:
count = 50
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama,John Q (2002),Drama
Save the Last Dance (2001),Romance,Saving Silverman (Evil Woman) (2001),Romance
"Wedding Planner, The (2001)",Comedy,National Lampoon's Van Wilder (2002),Comedy
"Wedding Planner, The (2001)",Romance,"Sweetest Thing, The (2002)",Romance
Hannibal (2001),Horror,Saw III (2006),Horror
Hannibal (2001),Thriller,"Sum of All Fears, The (2002)",Thriller
Saving Silverman (Evil Woman) (2001),Comedy,Cats & Dogs (2001),Comedy
Saving Silverman (Evil Woman) (2001),Romance,Save the Last Dance (2001),Romance
Down to Earth (2001),Comedy,Joe Dirt (2001),Comedy
Down to Earth (2001),Fantasy,"Haunted Mansion, The (2003)",Fantasy


结果表明，模型会推荐同一类型的电影。

<a name="4"></a>
## 4 - 恭喜！ <img align="left" src="./images/film_award.png" style=" width:40px;">
你已经完成了一个基于内容的推荐系统。   

这种结构是许多商业推荐系统的基础。如果有更多用户信息，用户内容可以大幅扩展以纳入这些信息。物品也不限于电影——它可以用来推荐任何物品：书籍、汽车，或者与你“购物车”中某件物品相似的物品。